# Stage 2 — Persian Watermark Robustness Test
**Project: Persian watermarking on speech synthesized with Mana-Persian-Piper + AudioSeal**

Stage 1b proved the watermark is detected 10 out of 10 on clean Mana speech.
But in the real world, audio never stays clean: it gets compressed, picks up noise, and is played faster or slower.
**Stage-2 question: how far does the watermark hold, and where does it break?**

**Attacks we test (on the watermarked file, for all 10 sentences):**

| Attack | Levels | Why does it matter? |
|---|---|---|
| MP3 | 256k, 128k, 64k, 32k | The most common audio compression on the internet |
| White noise | SNR 20dB, 10dB | Background noise, bad recording |
| Speed change | 0.9x, 1.1x | Fast/slow playback, video editing |
| Gain down | half (-6dB) | Making the signal quieter |
| Lowpass filter | 4kHz | Removing high frequencies (like a phone call) |
| 8kHz resampling | 16k → 8k → 16k | Telephone quality |

**We measure two numbers per attack:**
- **Detection:** mean watermark probability + bit accuracy (16 bits) — did the watermark survive?
- **Quality:** mean WER with Persian Whisper small — is the speech still intelligible?

**What does the "knee" mean?** The point where the watermark breaks: the first attack where the mean probability falls below 0.5 or bit accuracy drops below 90%. The last cell finds the knee and prints it.

**What do you send back?**
1. The file `summary_stage2.csv`
2. The zip file `persian-watermark-stage2.zip`
3. Answers: which attack broke the watermark first? Did MP3 at 32k break it or not?


## Cell 1 — Environment check: do we have a GPU?
As before: if there is no GPU, select `Runtime > Change runtime type > T4 GPU` from the top menu.


In [ ]:
import sys, torch
print("Python:", sys.version)
print("torch:", torch.__version__)
print("Is CUDA available?", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## Cell 2 — Install
`faster-whisper` is added to measure quality (WER), `scipy` for the lowpass filter.
It takes a few minutes. Wait for the green tick.


In [ ]:
!pip install -q persian-speech audioseal soundfile faster-whisper scipy 2>&1 | tail -3
print("Install done")


## Cell 3 — Check the install + silence torch warnings


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import torch
try:
    torch._dynamo.config.suppress_errors = True
except Exception as e:
    print("dynamo setting skipped:", e)
import persian_speech_runtime, audioseal, faster_whisper, scipy
print("All packages installed")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")


## Cell 4 — The same 10 fixed Persian sentences
Do not change the sentences or the 16-bit message — only then are the numbers of stages 1, 1b and 2 comparable.


In [ ]:
# The 10 fixed Persian test sentences — kept in Persian in every stage so the numbers stay comparable
SENTENCES = [
    "سلام، حالت چطور است؟",
    "هوا امروز در تهران آفتابی است.",
    "من هر روز صبح کتاب می‌خوانم.",
    "هوش مصنوعی دارد زندگی ما را تغییر می‌دهد.",
    "این آزمایش برای بررسی واترمارک فارسی انجام می‌شود.",
    "لطفا در را ببند و چراغ را خاموش کن.",
    "دانشجویان در آزمایشگاه روی پروژه گفتار کار می‌کنند.",
    "صدای ساخته‌شده باید طبیعی و قابل فهم باشد.",
    "ما می‌خواهیم بفهمیم واترمارک روی فارسی هم می‌ماند یا نه.",
    "فردا صبح ساعت نه به دانشگاه می‌روم.",
]
for i, s in enumerate(SENTENCES, 1):
    print(f"{i:02d}. {s}  ({len(s.split())} words)")
print()
print("Sentence count:", len(SENTENCES))
print("Total words:", sum(len(s.split()) for s in SENTENCES))


## Cell 5 — Load Mana-Persian-Piper
Exactly like stage 1b. Its output is 22050 Hz and is converted to 16000 in the next cell.


In [ ]:
from persian_speech_runtime import PiperSpeechEngine
tts = PiperSpeechEngine()
print("Mana-Persian-Piper ready.")
print("Note: this model's output is 22050 Hz; the next cell converts it to 16000.")


## Cell 6 — Single-sentence test: synthesize only sentence 1
If this one is broken, do not build the rest.


In [ ]:
import os, subprocess, soundfile as sf
from IPython.display import Audio, display

os.makedirs("persian_tts_clean", exist_ok=True)
os.makedirs("tmp_22050", exist_ok=True)

def synth_mana_16k(text, out_16k):
    tmp = out_16k.replace("persian_tts_clean", "tmp_22050").replace("_clean.wav", "_22050.wav")
    tts.save(text, tmp)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",tmp,"-ar","16000",out_16k], check=True)
    wav, sr = sf.read(out_16k)
    assert sr == 16000, f"Sample rate is {sr}, must be 16000"
    return wav, sr

wav1, sr = synth_mana_16k(SENTENCES[0], "persian_tts_clean/utt_001_clean.wav")
print("Sentence 1:", SENTENCES[0])
print("Audio length:", round(len(wav1)/sr, 2), "seconds | samples:", len(wav1), "| rate:", sr)
display(Audio(wav1, rate=sr))
print("Saved: persian_tts_clean/utt_001_clean.wav")


## Cell 7 — Synthesize all 10 clean sentences (16 kHz)
Output: `persian_tts_clean/utt_001_clean.wav` to `utt_010_clean.wav`


In [ ]:
import numpy as np, soundfile as sf, time
durations = []
t0 = time.time()
for i, text in enumerate(SENTENCES, 1):
    wav, sr = synth_mana_16k(text, f"persian_tts_clean/utt_{i:03d}_clean.wav")
    d = len(wav)/sr
    durations.append(d)
    print(f"{i:02d}. {d:.2f}s  persian_tts_clean/utt_{i:03d}_clean.wav")
print()
print(f"Total audio duration: {sum(durations):.2f} seconds")
print(f"Synthesis time (wall-clock): {time.time()-t0:.1f} seconds")


## Cell 8 — Load AudioSeal + helper functions
- Generator: `audioseal_wm_16bits` — 16-bit message
- Detector: `audioseal_detector_16bits`
- The fixed message is the same as stages 1 and 1b.


In [ ]:
import torch
from audioseal import AudioSeal
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading AudioSeal generator...")
generator = AudioSeal.load_generator("audioseal_wm_16bits").to(DEVICE)
generator.eval()
print("Loading AudioSeal detector...")
detector = AudioSeal.load_detector("audioseal_detector_16bits").to(DEVICE)
detector.eval()
print("AudioSeal ready.")
MESSAGE = torch.tensor([[1,0,1,1,0,1,0,0,1,1,0,1,0,1,1,0]], dtype=torch.int32, device=DEVICE)
print("Fixed 16-bit message:", MESSAGE.cpu().numpy().tolist()[0])

def to_tensor(wav_np):
    return torch.from_numpy(wav_np).float().unsqueeze(0).unsqueeze(0).to(DEVICE)

def detect_prob(wav_np):
    wav_t = to_tensor(wav_np)
    with torch.no_grad():
        try:
            result, msg = detector.detect_watermark(wav_t, sample_rate=16000)
        except TypeError:
            result, msg = detector.detect_watermark(wav_t)
    if isinstance(result, torch.Tensor):
        prob = float(result.detach().float().mean().cpu().item())
    else:
        prob = float(result)
    if isinstance(msg, torch.Tensor):
        bits = msg.detach().cpu().numpy().reshape(-1).astype(int).tolist()
    else:
        bits = list(msg)
    return prob, bits

def bit_accuracy(decoded):
    true = MESSAGE.cpu().numpy().reshape(-1).astype(int).tolist()
    correct = sum(int(a)==int(b) for a,b in zip(decoded, true))
    return correct, correct/len(true)

def embed_watermark(wav_np):
    with torch.no_grad():
        try:
            wm = generator.get_watermark(to_tensor(wav_np), sample_rate=16000, message=MESSAGE)
        except TypeError:
            wm = generator.get_watermark(to_tensor(wav_np), message=MESSAGE)
        return (to_tensor(wav_np) + wm).squeeze().cpu().numpy()


## Cell 9 — Watermark all 10 sentences + baseline
This is the same result as stage 1b, but it is rebuilt here so the notebook is self-contained and you do not need to upload the stage-1b zip.
The baseline should be 10 out of 10 as before — if it is not, stop and tell me.


In [ ]:
import soundfile as sf
os.makedirs("persian_tts_watermarked", exist_ok=True)
baseline = []
for i, text in enumerate(SENTENCES, 1):
    wav_clean, sr = sf.read(f"persian_tts_clean/utt_{i:03d}_clean.wav")
    p_clean, _ = detect_prob(wav_clean)
    wav_wm = embed_watermark(wav_clean)
    sf.write(f"persian_tts_watermarked/utt_{i:03d}_wm.wav", wav_wm, sr)
    p_wm, bits_wm = detect_prob(wav_wm)
    correct, acc = bit_accuracy(bits_wm)
    baseline.append({"id": i, "clean_prob": round(p_clean, 3), "wm_prob": round(p_wm, 3), "bits_correct": correct})
    print(f"{i:02d}. clean={p_clean:.3f}  watermarked={p_wm:.3f}  bits={correct}/16")
print()
print("Baseline (no attack) ready — should be 10/10 like step 1b.")


## Cell 10 — Define the attacks
Each attack is a function that takes the watermarked audio and returns the attacked version:
- **MP3:** encoded and decoded with ffmpeg (bitrate is a parameter).
- **Noise:** Gaussian white noise is added; its strength is set by SNR (20 dB = little, 10 dB = a lot). A fixed seed keeps it reproducible.
- **Speed:** ffmpeg's atempo filter; 0.9 means slower and longer, 1.1 means faster and shorter.
- **Gain down:** multiply by 0.5 (minus 6 dB).
- **Lowpass filter:** 4th-order Butterworth with a 4 kHz cutoff.
- **8 kHz resampling:** round trip 16 → 8 → 16 kHz (telephone quality).


In [ ]:
import subprocess, numpy as np, soundfile as sf
from scipy.signal import butter, sosfiltfilt

def attack_mp3(wav_np, sr, bitrate_kbps, tmp_in="/tmp/atk_in.wav", tmp_mp3="/tmp/atk.mp3", tmp_out="/tmp/atk_out.wav"):
    sf.write(tmp_in, wav_np, sr)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",tmp_in,"-codec:a","libmp3lame","-b:a",f"{bitrate_kbps}k",tmp_mp3], check=True)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",tmp_mp3,"-ar","16000",tmp_out], check=True)
    wav, sr2 = sf.read(tmp_out)
    assert sr2 == 16000
    return wav.astype(np.float32)

def attack_noise(wav_np, sr, snr_db, seed):
    rng = np.random.default_rng(seed)
    sig_rms = float(np.sqrt(np.mean(wav_np**2))) + 1e-12
    noise_rms = sig_rms / (10 ** (snr_db / 20))
    noise = rng.normal(0, noise_rms, size=wav_np.shape).astype(np.float32)
    return np.clip(wav_np + noise, -1.0, 1.0).astype(np.float32)

def attack_speed(wav_np, sr, factor, tmp_in="/tmp/atk_in.wav", tmp_out="/tmp/atk_out.wav"):
    sf.write(tmp_in, wav_np, sr)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",tmp_in,"-filter:a",f"atempo={factor}","-ar","16000",tmp_out], check=True)
    wav, sr2 = sf.read(tmp_out)
    assert sr2 == 16000
    return wav.astype(np.float32)

def attack_gain(wav_np, sr, factor):
    return np.clip(wav_np * factor, -1.0, 1.0).astype(np.float32)

def attack_lowpass(wav_np, sr, cutoff_hz):
    sos = butter(4, cutoff_hz, btype="low", fs=sr, output="sos")
    return sosfiltfilt(sos, wav_np).astype(np.float32)

def attack_resample8k(wav_np, sr, tmp_in="/tmp/atk_in.wav", tmp_mid="/tmp/atk_8k.wav", tmp_out="/tmp/atk_out.wav"):
    sf.write(tmp_in, wav_np, sr)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",tmp_in,"-ar","8000",tmp_mid], check=True)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",tmp_mid,"-ar","16000",tmp_out], check=True)
    wav, sr2 = sf.read(tmp_out)
    assert sr2 == 16000
    return wav.astype(np.float32)

ATTACKS = [
    ("mp3_256",    "MP3 256k",            lambda w, sr, i: attack_mp3(w, sr, 256)),
    ("mp3_128",    "MP3 128k",            lambda w, sr, i: attack_mp3(w, sr, 128)),
    ("mp3_64",     "MP3 64k",             lambda w, sr, i: attack_mp3(w, sr, 64)),
    ("mp3_32",     "MP3 32k",             lambda w, sr, i: attack_mp3(w, sr, 32)),
    ("noise_20db", "Noise SNR 20dB",      lambda w, sr, i: attack_noise(w, sr, 20, seed=1000+i)),
    ("noise_10db", "Noise SNR 10dB",      lambda w, sr, i: attack_noise(w, sr, 10, seed=1000+i)),
    ("speed_0.9",  "Speed 0.9x",          lambda w, sr, i: attack_speed(w, sr, 0.9)),
    ("speed_1.1",  "Speed 1.1x",          lambda w, sr, i: attack_speed(w, sr, 1.1)),
    ("gain_0.5",   "Gain 0.5 (-6dB)",     lambda w, sr, i: attack_gain(w, sr, 0.5)),
    ("lowpass_4k", "Lowpass 4kHz",        lambda w, sr, i: attack_lowpass(w, sr, 4000)),
    ("resample_8k","Resample 8kHz",       lambda w, sr, i: attack_resample8k(w, sr)),
]
print(f"Attack count: {len(ATTACKS)}")
for name, label, _ in ATTACKS:
    print(f" - {name}: {label}")


## Cell 11 — Apply all attacks to all 10 sentences
Output: `persian_wm_attacked/<attack>/utt_001_wm.wav` to `utt_010_wm.wav` — that is 11 attacks × 10 sentences = 110 files.


In [ ]:
import os, soundfile as sf, time
os.makedirs("persian_wm_attacked", exist_ok=True)
t0 = time.time()
for name, label, fn in ATTACKS:
    os.makedirs(f"persian_wm_attacked/{name}", exist_ok=True)
    for i in range(1, 11):
        wav_wm, sr = sf.read(f"persian_tts_watermarked/utt_{i:03d}_wm.wav")
        wav_atk = fn(wav_wm, sr, i)
        sf.write(f"persian_wm_attacked/{name}/utt_{i:03d}_wm.wav", wav_atk, 16000)
    print(f"{name} ({label}): 10 files created")
print()
print(f"Total attacked files: {len(ATTACKS)*10} files in {time.time()-t0:.1f} seconds")


## Cell 12 — Detect the watermark on all attacked files
Per attack: mean probability, how many of the 10 sentences were detected, mean bit accuracy.


In [ ]:
import soundfile as sf
rows = []
for name, label, _ in ATTACKS:
    probs, accs = [], []
    for i in range(1, 11):
        wav_atk, sr = sf.read(f"persian_wm_attacked/{name}/utt_{i:03d}_wm.wav")
        p, bits = detect_prob(wav_atk)
        correct, acc = bit_accuracy(bits)
        rows.append({"id": i, "attack": name, "attack_label": label, "wm_prob": round(p, 3), "bits_correct": correct, "bit_accuracy": round(acc, 3), "detected_gt_0.5": bool(p > 0.5)})
        probs.append(p); accs.append(acc)
    print(f"{name:12s} mean prob={np.mean(probs):.3f}  detected={int(sum(p > 0.5 for p in probs))}/10  mean bit acc={np.mean(accs):.2%}")
print()
print("Detection on all attacks done.")


## Cell 13 — Listen: sentence 1 after the attacks
Before the quality numbers, listen yourself: how badly do MP3 at 32k and noise at 10dB damage the audio?


In [ ]:
from IPython.display import Audio, display
import soundfile as sf
print("— Sentence 1, watermarked, no attack —")
wav, sr = sf.read("persian_tts_watermarked/utt_001_wm.wav"); display(Audio(wav, rate=sr))
print("— Sentence 1, after MP3 32k —")
wav, sr = sf.read("persian_wm_attacked/mp3_32/utt_001_wm.wav"); display(Audio(wav, rate=sr))
print("— Sentence 1, after noise 10dB —")
wav, sr = sf.read("persian_wm_attacked/noise_10db/utt_001_wm.wav"); display(Audio(wav, rate=sr))
print("— Sentence 1, after speed 1.1x —")
wav, sr = sf.read("persian_wm_attacked/speed_1.1/utt_001_wm.wav"); display(Audio(wav, rate=sr))


## Cell 14 — Quality: WER with Persian Whisper small
For every condition (no attack + 11 attacks) we transcribe all 10 sentences with Whisper small and compare with the original sentence.
WER means "word error rate": what percentage of words were misheard. Low WER = the speech is still intelligible.
To keep the comparison fair, we convert the half-space to a space and turn Arabic «ي/ك» back into Persian «ی/ک» — in the reference text and in Whisper's output alike.

**Important note:** we run Whisper on CPU on purpose, because faster-whisper on Colab GPU errors with `libcublas.so.12` missing. The small model is small and fast enough on CPU.

**Why small and not tiny?** In the first run with tiny, WER was about 95–100% everywhere — even 93% with no attack — because tiny does not understand Persian well and heard sentence 1 as «سالوم، حالا چی توراس». So the WER column with tiny is useless for comparing attacks; small is more accurate and shows the real range.


In [ ]:
from faster_whisper import WhisperModel
# Note: we run Whisper on CPU on purpose.
# faster-whisper on Colab GPU cannot find the libcublas.so.12 library and errors out.
# The small model is fast enough on CPU; AudioSeal stays on GPU.
WHISPER_DEVICE = "cpu"
print("Loading Whisper small on CPU ...")
whisper = WhisperModel("small", device=WHISPER_DEVICE, compute_type="int8")
print("Whisper small ready.")

def normalize_fa(t):
    t = t.replace("ي", "ی").replace("ك", "ک").replace("\u200c", " ")
    for ch in "؟،.,!؛:()«»\"'":
        t = t.replace(ch, " ")
    return t.split()

def wer(ref, hyp):
    r, h = normalize_fa(ref), normalize_fa(hyp)
    n, m = len(r), len(h)
    if n == 0:
        return 1.0 if m else 0.0
    d = list(range(m + 1))
    for i in range(1, n + 1):
        prev, d[0] = d[0], i
        for j in range(1, m + 1):
            cur = d[j]
            d[j] = min(d[j] + 1, d[j - 1] + 1, prev + (0 if r[i - 1] == h[j - 1] else 1))
            prev = cur
    return d[m] / n

def transcribe_wer(wav_np, ref_text):
    segments, _ = whisper.transcribe(wav_np.astype(np.float32), language="fa", beam_size=1)
    hyp = "".join(s.text for s in segments).strip()
    return wer(ref_text, hyp), hyp

print("WER test on sentence 1, no attack:")
wav, sr = sf.read("persian_tts_watermarked/utt_001_wm.wav")
w, hyp = transcribe_wer(wav, SENTENCES[0])
print("Reference:    ", SENTENCES[0])
print("Heard:", hyp)
print(f"WER, sentence 1: {w:.2%}")


## Cell 15 — Compute WER for all conditions
12 conditions (no attack + 11 attacks) × 10 sentences = 120 transcriptions. On CPU it takes about 15 to 30 minutes (the small model is bigger and more accurate than tiny).


In [ ]:
import soundfile as sf, time
wer_rows = []
t0 = time.time()
conditions = [("clean_wm", "No attack", "persian_tts_watermarked")] + [(name, label, f"persian_wm_attacked/{name}") for name, label, _ in ATTACKS]
for cond_name, cond_label, folder in conditions:
    wers = []
    for i in range(1, 11):
        wav, sr = sf.read(f"{folder}/utt_{i:03d}_wm.wav")
        w, hyp = transcribe_wer(wav, SENTENCES[i - 1])
        wer_rows.append({"id": i, "attack": cond_name, "wer": round(w, 3)})
        wers.append(w)
    print(f"{cond_name:12s} mean WER={np.mean(wers):.2%}")
print()
print(f"All WERs computed in {time.time()-t0:.1f} seconds.")


## Cell 16 — Stage-2 summary table
Detection + WER go into one file: `summary_stage2.csv` — each row is one sentence under one attack.


In [ ]:
import pandas as pd
det_df = pd.DataFrame(rows)
wer_df = pd.DataFrame(wer_rows)
summary = det_df.merge(wer_df, on=["id", "attack"], how="left")
base_rows = []
for b in baseline:
    w = wer_df[(wer_df["id"] == b["id"]) & (wer_df["attack"] == "clean_wm")]["wer"].values
    base_rows.append({"id": b["id"], "attack": "clean_wm", "attack_label": "No attack", "wm_prob": b["wm_prob"], "bits_correct": b["bits_correct"], "bit_accuracy": round(b["bits_correct"] / 16, 3), "detected_gt_0.5": bool(b["wm_prob"] > 0.5), "wer": float(w[0]) if len(w) else None})
summary = pd.concat([pd.DataFrame(base_rows), summary], ignore_index=True)
summary.to_csv("summary_stage2.csv", index=False, encoding="utf-8")
print("Saved: summary_stage2.csv")
pivot = summary.groupby(["attack", "attack_label"]).agg(mean_prob=("wm_prob", "mean"), detected=("detected_gt_0.5", "sum"), mean_bits=("bit_accuracy", "mean"), mean_wer=("wer", "mean")).reset_index()
print(pivot.to_string(index=False))


## Cell 17 — Stage-2 charts
Four panels (labels are in English so the matplotlib font does not break):
- Top-left: mean detection probability per attack (dashed line = 0.5 threshold)
- Top-right: mean bit accuracy per attack
- Bottom-left: mean WER per attack (audio quality)
- Bottom-right: MP3 knee curve — detection probability vs bitrate


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd, numpy as np
summary = pd.read_csv("summary_stage2.csv")
pivot = summary.groupby(["attack", "attack_label"]).agg(mean_prob=("wm_prob", "mean"), mean_bits=("bit_accuracy", "mean"), mean_wer=("wer", "mean")).reset_index()
order = ["clean_wm"] + [a[0] for a in ATTACKS]
pivot["ord"] = pivot["attack"].apply(lambda a: order.index(a))
pivot = pivot.sort_values("ord")
labels = pivot["attack"].tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
x = np.arange(len(pivot))
axes[0, 0].bar(x, pivot["mean_prob"]); axes[0, 0].axhline(0.5, linestyle="--")
axes[0, 0].set_xticks(x); axes[0, 0].set_xticklabels(labels, rotation=45, ha="right")
axes[0, 0].set_ylabel("Mean detection probability"); axes[0, 0].set_title("Detection probability per attack (threshold 0.5)")
axes[0, 0].set_ylim(0, 1.05); axes[0, 0].grid(axis="y", alpha=0.3)
axes[0, 1].bar(x, pivot["mean_bits"])
axes[0, 1].set_xticks(x); axes[0, 1].set_xticklabels(labels, rotation=45, ha="right")
axes[0, 1].set_ylabel("Mean bit accuracy"); axes[0, 1].set_title("Bit accuracy per attack (of 16 bits)")
axes[0, 1].set_ylim(0, 1.05); axes[0, 1].grid(axis="y", alpha=0.3)
axes[1, 0].bar(x, pivot["mean_wer"])
axes[1, 0].set_xticks(x); axes[1, 0].set_xticklabels(labels, rotation=45, ha="right")
axes[1, 0].set_ylabel("Mean WER"); axes[1, 0].set_title("Whisper small WER per attack (quality)")
axes[1, 0].set_ylim(0, 1.05); axes[1, 0].grid(axis="y", alpha=0.3)
mp3 = pivot[pivot["attack"].str.startswith("mp3")].copy()
mp3["kbps"] = mp3["attack"].str.replace("mp3_", "").astype(int)
mp3 = mp3.sort_values("kbps")
axes[1, 1].plot(mp3["kbps"], mp3["mean_prob"], marker="o")
axes[1, 1].axhline(0.5, linestyle="--")
axes[1, 1].set_xscale("log"); axes[1, 1].set_xticks(mp3["kbps"]); axes[1, 1].get_xaxis().set_major_formatter(plt.ScalarFormatter())
axes[1, 1].set_xlabel("MP3 bitrate (kbps)"); axes[1, 1].set_ylabel("Mean detection probability")
axes[1, 1].set_title("MP3 knee: detection vs bitrate"); axes[1, 1].set_ylim(0, 1.05); axes[1, 1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig("chart_stage2.png", dpi=150); plt.show()
print("Saved: chart_stage2.png")


## Cell 18 — Where is the knee?
Our definition: an attack is "broken" if the mean probability falls below 0.5 or bit accuracy drops below 90%.
This cell prints the first breaking point for each attack family.


In [ ]:
print("=== Breaking point (knee) ===")
print("Definition: mean probability below 0.5, or bit accuracy below 90%, means broken.")
print()
for fam, names in [("MP3", ["mp3_256", "mp3_128", "mp3_64", "mp3_32"]), ("Noise", ["noise_20db", "noise_10db"]), ("Speed", ["speed_0.9", "speed_1.1"])]:
    print(f"— {fam}:")
    for n in names:
        r = pivot[pivot["attack"] == n].iloc[0]
        status = "intact" if r["mean_prob"] > 0.5 and r["mean_bits"] >= 0.9 else "broken"
        print(f"   {n:12s} prob={r['mean_prob']:.3f}  bit acc={r['mean_bits']:.2%}  WER={r['mean_wer']:.2%}  {status}")
print()
print("— Single attacks:")
for n in ["gain_0.5", "lowpass_4k", "resample_8k"]:
    r = pivot[pivot["attack"] == n].iloc[0]
    status = "intact" if r["mean_prob"] > 0.5 and r["mean_bits"] >= 0.9 else "broken"
    print(f"   {n:12s} prob={r['mean_prob']:.3f}  bit acc={r['mean_bits']:.2%}  WER={r['mean_wer']:.2%}  {status}")


## Cell 19 — Zip the outputs for download
The zip contains 10 watermarked files + 110 attacked files + the summary table + the chart.
It does not include the clean files — you already have them from the stage-1b zip.


In [ ]:
import os, zipfile
with zipfile.ZipFile("persian-watermark-stage2.zip", "w") as z:
    for fn in sorted(os.listdir("persian_tts_watermarked")):
        z.write(os.path.join("persian_tts_watermarked", fn))
    for name, _, _ in ATTACKS:
        for fn in sorted(os.listdir(f"persian_wm_attacked/{name}")):
            z.write(os.path.join(f"persian_wm_attacked/{name}", fn))
    z.write("summary_stage2.csv")
    if os.path.exists("chart_stage2.png"):
        z.write("chart_stage2.png")
print("Created: persian-watermark-stage2.zip")
print("Size:", round(os.path.getsize("persian-watermark-stage2.zip") / 1024 / 1024, 1), "MB")
print("Files: 10 watermarked + 110 attacked + summary_stage2.csv + chart_stage2.png")


## Done — what do you send back?
**1. Send me the file `summary_stage2.csv`**
**2. Send me the file `persian-watermark-stage2.zip` too**

**And answer these questions:**
- **A)** Which attack broke the watermark first? (Read it from the output of cell 18)
- **B)** Did MP3 at 32k break it or not? What about noise at 10dB?
- **C)** Listening: was sentence 1 still intelligible after MP3 at 32k? What about after noise at 10dB?

**If stage 2 is complete, the next steps are (not chosen yet):**
- Compare with WavMark on the same 10 sentences and the same attacks
- Or test harder attacks: cropping, echo, Opus compression
- Or write the final Persian report with these same numbers
